# Features

Punto 3 de `docs/decisions.md`. Arma `data/processed/features.parquet` desde la foto de `03_dataset.ipynb` (una fila por merchant).

- **25 features:** 21 originales + 3 relativas a la industria + `max_rechazos_1d_30d`, el peor día de rechazos en los 30 días que terminan en la foto. Ese pico no cae el día de la foto en `M7F910` (día −3) ni en `MC45B9` (día −21).
- Sin lags ni diferencias: las ventanas de 7, 15 y 30 días ya están en la foto.
- En las tasas, el valor −1 (sin transacciones) se toma como nulo.
- `group_d` queda vacío: `MEA1EE5` nunca operó y no entra al dataset. La columna no es una feature.

La lista sale del análisis feature por feature del panel diario anterior. Ese análisis no se vuelve a correr: `dataset.parquet` ya no es un panel. Los motivos de cada columna que queda afuera están en `docs/decisions.md`.

Salida: `data/processed/features.parquet` con `merchant_id`, `observation_date`, `target`, `group_d` y las 25 features.

In [ ]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
MERCHANT_PATH = ROOT / "data" / "merchant_rating_featureset_sample.parquet"
DATASET_PATH = ROOT / "data" / "processed" / "dataset.parquet"
OUT_PATH = ROOT / "data" / "processed" / "features.parquet"
KEYS = ["merchant_id", "observation_date"]

BASE_FEATURES = [
    # aprobación actual
    "me_txn_approval_rate_1d", "me_txn_approval_rate_7d", "me_txn_approval_rate_15d", "me_txn_approval_rate_30d",
    "me_tpv_approval_rate_1d", "me_tpv_approval_rate_7d", "me_tpv_approval_rate_15d", "me_tpv_approval_rate_30d",
    # aprobación histórica
    "me_lifetime_txn_approval_rate", "me_lifetime_tpv_approval_rate",
    # ticket
    "me_attempted_tpv_avg_1d", "me_attempted_tpv_avg_7d", "me_attempted_tpv_avg_30d", "me_lifetime_tpv_avg",
    # intentos
    "me_attempted_txn_sum_1d", "me_attempted_txn_sum_7d", "me_attempted_txn_sum_15d", "me_attempted_txn_sum_30d",
    # ecommerce y tamaño
    "me_attempted_ecommerce_txn_sum_7d", "approved_tpv_90d", "me_lifetime_approved_txn",
]
# Insumos de las 3 features relativas a la industria; no quedan como columnas.
INDUSTRY_SOURCES = ["approved_txns_90d", "p50_approved_tpv_90d", "p95_approved_tpv_90d", "p50_approved_txns_90d"]

# Fuera por definición (paso 1 del punto 3): fecha, vacía, constante, duplicadas y fuga.
EXCLUDED_BY_DEFINITION = [
    "me_merchant_created_at_timestamp", "config_segment", "me_attempted_pocket_txn_sum_7d",
    "me_industry_mcc", "vip0_segment",
    "me_suspended_merchants_in_network", "me_suspended_merchants_in_network_share",
]

## Foto del día elegido

Los valores de cada merchant en la fecha que eligió `03_dataset.ipynb`.

In [ ]:
dataset = pd.read_parquet(DATASET_PATH, columns=KEYS + ["target", "group_d"])
dataset["observation_date"] = pd.to_datetime(dataset["observation_date"])
dates = sorted(dataset["observation_date"].dt.date.unique())

daily = pd.read_parquet(
    MERCHANT_PATH,
    columns=KEYS + BASE_FEATURES + INDUSTRY_SOURCES,
    filters=[("observation_date", "in", dates)],
)
daily["observation_date"] = pd.to_datetime(daily["observation_date"])
rate_cols = [c for c in daily.columns if "rate" in c]
daily[rate_cols] = daily[rate_cols].where(daily[rate_cols] > -0.5)  # -1 = sin transacciones

features = dataset.merge(daily, on=KEYS, how="left", validate="one_to_one")
print(f"{len(features):,} filas")

## Relativas a la industria

El merchant contra su industria, en vez de las variables de industria en crudo (con 12 positivos el modelo memorizaría MCC):

- `tpv_vs_industry_p50` = TPV aprobado a 90 días / mediana de su industria.
- `tpv_vs_industry_p95` = TPV aprobado a 90 días / percentil 95 de su industria.
- `txns_vs_industry_p50` = transacciones aprobadas a 90 días / mediana de su industria.

In [ ]:
def ratio(num, den):
    return (num / den.where(den > 0)).astype("float64")


features["tpv_vs_industry_p50"] = ratio(features["approved_tpv_90d"], features["p50_approved_tpv_90d"])
features["tpv_vs_industry_p95"] = ratio(features["approved_tpv_90d"], features["p95_approved_tpv_90d"])
features["txns_vs_industry_p50"] = ratio(features["approved_txns_90d"], features["p50_approved_txns_90d"])
features = features.drop(columns=INDUSTRY_SOURCES)

## Guardar

In [ ]:
# Peor día de rechazos en los 30 días hasta la foto, incluido ese día.
history = pd.read_parquet(
    MERCHANT_PATH,
    columns=["merchant_id", "observation_date", "me_attempted_txn_sum_1d", "me_approved_txn_sum_1d"],
    filters=[("merchant_id", "in", features["merchant_id"].tolist())],
)
history["observation_date"] = pd.to_datetime(history["observation_date"])
attempted_day = history["me_attempted_txn_sum_1d"].where(history["me_attempted_txn_sum_1d"] > -0.5)
approved_day = history["me_approved_txn_sum_1d"].where(history["me_approved_txn_sum_1d"] > -0.5)
history["rej"] = (attempted_day - approved_day).where(attempted_day.notna()).fillna(0)
window = history.merge(
    features[KEYS].rename(columns={"observation_date": "foto"}),
    on="merchant_id", how="inner",
)
window = window[
    (window["observation_date"] <= window["foto"])
    & (window["observation_date"] >= window["foto"] - pd.Timedelta(days=30))
]
features["max_rechazos_1d_30d"] = features["merchant_id"].map(window.groupby("merchant_id")["rej"].max()).fillna(0)

feature_cols = [c for c in features.columns if c not in KEYS + ["target", "group_d"]]
print(f"{len(feature_cols)} features")
assert len(feature_cols) == 25
assert not set(feature_cols) & set(EXCLUDED_BY_DEFINITION)
assert len(features) == len(dataset)

nulls = features[feature_cols].isna().mean().sort_values(ascending=False)
print("nulos por feature (las 10 con más):")
print((nulls.head(10) * 100).round(1).to_string())

OUT_PATH.parent.mkdir(exist_ok=True)
features.to_parquet(OUT_PATH, index=False)
print("guardado", OUT_PATH)